# Clustering Algorithms in TuiML

This tutorial covers unsupervised clustering algorithms for grouping similar data points.

In [1]:
# Common imports from TuiML
import numpy as np
from tuiml.evaluation import (
    silhouette_score,
    adjusted_rand_score
)
from tuiml.preprocessing import StandardScaler
from tuiml.datasets import load_iris, load_glass

## 1. K-Means Clustering (KMeansClusterer)

The classic partitioning algorithm that divides data into k clusters.

In [2]:
from tuiml.algorithms.clustering import KMeansClusterer

# Load dataset
X, y_true = load_iris()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# K-Means Clusterer
kmeans = KMeansClusterer(n_clusters=3, random_state=42, max_iter=300)
labels = kmeans.fit_predict(X_scaled)

print("KMeansClusterer (k=3):")
print(f"  Silhouette Score: {silhouette_score(X_scaled, labels):.4f}")
print(f"  Adjusted Rand Index: {adjusted_rand_score(y_true, labels):.4f}")
print(f"  Cluster centers shape: {kmeans.cluster_centers_.shape}")

KMeansClusterer (k=3):
  Silhouette Score: 0.4590
  Adjusted Rand Index: 0.6201
  Cluster centers shape: (3, 4)


In [3]:
# Finding optimal k using Elbow method
print("\nFinding optimal k (Elbow method):")
for k in range(2, 8):
    km = KMeansClusterer(n_clusters=k, random_state=42)
    labels = km.fit_predict(X_scaled)
    sil_score = silhouette_score(X_scaled, labels)
    print(f"  k={k}: Silhouette={sil_score:.4f}, Inertia={km.inertia_:.2f}")


Finding optimal k (Elbow method):
  k=2: Silhouette=0.5802, Inertia=223.73
  k=3: Silhouette=0.4590, Inertia=140.97
  k=4: Silhouette=0.3979, Inertia=114.71
  k=5: Silhouette=0.3477, Inertia=91.07
  k=6: Silhouette=0.3454, Inertia=81.74
  k=7: Silhouette=0.3310, Inertia=71.81


## 2. Hierarchical Clustering

Builds a tree of clusters using agglomerative (bottom-up) approach.

In [4]:
from tuiml.algorithms.clustering import AgglomerativeClusterer

# Agglomerative Clustering (Hierarchical)
hc = AgglomerativeClusterer(n_clusters=3, linkage='ward')
labels = hc.fit_predict(X_scaled)

print("AgglomerativeClusterer (Ward linkage):")
print(f"  Silhouette Score: {silhouette_score(X_scaled, labels):.4f}")
print(f"  Adjusted Rand Index: {adjusted_rand_score(y_true, labels):.4f}")

AgglomerativeClusterer (Ward linkage):
  Silhouette Score: 0.4455
  Adjusted Rand Index: 0.6153


In [5]:
# Try different linkage methods
print("\nComparing linkage methods:")
for linkage in ['single', 'complete', 'average', 'ward']:
    hc = AgglomerativeClusterer(n_clusters=3, linkage=linkage)
    labels = hc.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    ari = adjusted_rand_score(y_true, labels)
    print(f"  {linkage:<10}: Silhouette={sil:.4f}, ARI={ari:.4f}")


Comparing linkage methods:
  single    : Silhouette=0.5029, ARI=0.5584
  complete  : Silhouette=0.4488, ARI=0.5726
  average   : Silhouette=0.4795, ARI=0.5621
  ward      : Silhouette=0.4455, ARI=0.6153


## 3. DBSCAN (Density-Based Clustering)

Finds clusters based on density. Can discover arbitrarily shaped clusters and identify outliers.

In [6]:
from tuiml.algorithms.clustering import DBSCANClusterer

# DBSCAN Clusterer
dbscan = DBSCANClusterer(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print("DBSCANClusterer:")
print(f"  Number of clusters: {n_clusters}")
print(f"  Noise points: {n_noise}")

# Only compute silhouette if we have at least 2 clusters and non-noise points
if n_clusters >= 2:
    # Exclude noise points for silhouette
    mask = labels != -1
    if mask.sum() > n_clusters:
        sil = silhouette_score(X_scaled[mask], labels[mask])
        print(f"  Silhouette Score: {sil:.4f}")

DBSCANClusterer:
  Number of clusters: 2
  Noise points: 35
  Silhouette Score: 0.6532


In [7]:
# Tune DBSCANClusterer parameters
print("\nDBSCANClusterer parameter tuning:")
for eps in [0.3, 0.5, 0.7, 1.0]:
    db = DBSCANClusterer(eps=eps, min_samples=5)
    labels = db.fit_predict(X_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    print(f"  eps={eps}: clusters={n_clusters}, noise={n_noise}")


DBSCANClusterer parameter tuning:
  eps=0.3: clusters=3, noise=120
  eps=0.5: clusters=2, noise=35
  eps=0.7: clusters=2, noise=8
  eps=1.0: clusters=2, noise=3


## 4. Expectation-Maximization (GaussianMixtureClusterer)

Probabilistic clustering using Gaussian Mixture Models.

In [9]:
from tuiml.algorithms.clustering import GaussianMixtureClusterer

# GaussianMixtureClusterer (EM)
em = GaussianMixtureClusterer(n_components=3, max_iter=100, random_state=42)
labels = em.fit_predict(X_scaled)

print("GaussianMixtureClusterer:")
print(f"  Silhouette Score: {silhouette_score(X_scaled, labels):.4f}")
print(f"  Adjusted Rand Index: {adjusted_rand_score(y_true, labels):.4f}")

# Get soft cluster assignments (probabilities)
proba = em.predict_proba(X_scaled[:5])
print(f"\nCluster probabilities (first 5 samples):")
print(proba)

GaussianMixtureClusterer:
  Silhouette Score: 0.3018
  Adjusted Rand Index: 0.6007

Cluster probabilities (first 5 samples):
[[1.47630931e-10 1.00000000e+00 3.66832344e-20]
 [1.91965347e-07 9.99999808e-01 8.14811275e-16]
 [5.13260290e-08 9.99999949e-01 2.52070907e-17]
 [1.02119128e-07 9.99999898e-01 2.09751865e-14]
 [6.44587077e-11 1.00000000e+00 1.19449767e-20]]


## 5. Canopy Clustering

Fast pre-clustering algorithm using loose/tight thresholds.

In [10]:
from tuiml.algorithms.clustering import CanopyClusterer

# Canopy Clustering
canopy = CanopyClusterer(t1=3.0, t2=1.5)  # t1 > t2 (loose > tight threshold)
labels = canopy.fit_predict(X_scaled)

n_clusters = len(set(labels))
print(f"CanopyClusterer:")
print(f"  Number of canopies: {n_clusters}")

if n_clusters >= 2:
    print(f"  Silhouette Score: {silhouette_score(X_scaled, labels):.4f}")

CanopyClusterer:
  Number of canopies: 10
  Silhouette Score: 0.3004


## 6. FarthestFirst Clustering

Greedy algorithm that initializes clusters as far apart as possible.

In [11]:
from tuiml.algorithms.clustering import FarthestFirstClusterer

# FarthestFirst Clusterer
ff = FarthestFirstClusterer(n_clusters=3, random_state=42)
labels = ff.fit_predict(X_scaled)

print("FarthestFirstClusterer:")
print(f"  Silhouette Score: {silhouette_score(X_scaled, labels):.4f}")
print(f"  Adjusted Rand Index: {adjusted_rand_score(y_true, labels):.4f}")

FarthestFirstClusterer:
  Silhouette Score: 0.4281
  Adjusted Rand Index: 0.5758


## 7. Cobweb (Conceptual Clustering)

In [12]:
from tuiml.algorithms.clustering import CobwebClusterer

# Cobweb (incremental conceptual clustering)
cobweb = CobwebClusterer(cutoff=0.1, acuity=1.0)
labels = cobweb.fit_predict(X_scaled)

n_clusters = len(set(labels))
print(f"CobwebClusterer:")
print(f"  Number of clusters: {n_clusters}")
if n_clusters >= 2:
    print(f"  Silhouette Score: {silhouette_score(X_scaled, labels):.4f}")

CobwebClusterer:
  Number of clusters: 3
  Silhouette Score: -0.0347


## 8. Comparing Clustering Algorithms

In [13]:
# Compare algorithms on Glass dataset
X_glass, y_glass = load_glass()
scaler = StandardScaler()
X_glass_scaled = scaler.fit_transform(X_glass)

# Number of true classes
n_true_clusters = len(set(y_glass))
print(f"Glass dataset: {n_true_clusters} classes\n")

algorithms = [
    ('KMeansClusterer', KMeansClusterer(n_clusters=n_true_clusters, random_state=42)),
    ('AgglomerativeClusterer', AgglomerativeClusterer(n_clusters=n_true_clusters, linkage='ward')),
    ('GaussianMixtureClusterer', GaussianMixtureClusterer(n_components=n_true_clusters, random_state=42)),
    ('FarthestFirstClusterer', FarthestFirstClusterer(n_clusters=n_true_clusters, random_state=42)),
]

print("Clustering Algorithm Comparison:")
print("=" * 55)

for name, alg in algorithms:
    labels = alg.fit_predict(X_glass_scaled)
    sil = silhouette_score(X_glass_scaled, labels)
    ari = adjusted_rand_score(y_glass, labels)
    print(f"{name:<25}: Silhouette={sil:.4f}, ARI={ari:.4f}")

Glass dataset: 6 classes

Clustering Algorithm Comparison:
KMeansClusterer          : Silhouette=0.3276, ARI=0.1642
AgglomerativeClusterer   : Silhouette=0.2822, ARI=0.1351
GaussianMixtureClusterer : Silhouette=0.1690, ARI=0.1926
FarthestFirstClusterer   : Silhouette=0.4799, ARI=0.0143


## 9. Distance Functions

TuiML provides various distance functions for clustering.

In [14]:
from tuiml.algorithms.clustering import (
    euclidean_distance,
    manhattan_distance,
    cosine_distance,
    pairwise_distances
)

# Example points
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("Distance between [1,2,3] and [4,5,6]:")
print(f"  Euclidean: {euclidean_distance(a, b):.4f}")
print(f"  Manhattan: {manhattan_distance(a, b):.4f}")
print(f"  Cosine: {cosine_distance(a, b):.4f}")

# Pairwise distances
X_sample = X_scaled[:5]
dist_matrix = pairwise_distances(X_sample, metric='euclidean')
print(f"\nPairwise distance matrix shape: {dist_matrix.shape}")

Distance between [1,2,3] and [4,5,6]:
  Euclidean: 5.1962
  Manhattan: 9.0000
  Cosine: 0.0254

Pairwise distance matrix shape: (5, 5)
